# worm6 CapGTA GEX preprocessing sweep

The CapGTA RNA channel is dominated by amplification artifact — technical
variation between cells (WGA duplication of a random subset of transcripts)
swamps biological signal.

This notebook starts from **raw counts** (before any normalization / scaling)
for each read-source × SoupX combination we have on disk, then sweeps
downstream preprocessing arms and, for each config, reports:

1. UMAP colored by leiden clusters
2. Top-100 DEGs by fold change per cluster (saved as CSV)
3. |Pearson r| between the first 30 PCs and two DNA library-size metrics
   (`pct_1x`, `unique_aligned_reads`) — the tighter the PCs track DNA
   coverage, the more the RNA channel is being driven by WGA artifact

**Sweep dimensions**
- **Read source**: `spliced` (CIGAR ~ N), `splicedpolyat` (∪ ≥10 A/T run), `polyat` (≥10 A/T run only)
- **SoupX**: off / on
- **Count scaling** applied to `.X` before normalization:
  `none`, `/n_junctions`, `/exonic_length_kb`, `/(J·L_kb)` (the arm-B "current" scaling in `preprocess_scanpy.py`)
- **Normalization**: standard `normalize_total(target_sum=1e4)`, or divide counts by DNA size factor (`unique_aligned_reads / median`) — both followed by `log1p`
- **Regression**: none, regress out `pct_1x`, regress out `log(unique_aligned_reads+1)`

Raw counts live in `.layers['counts']` throughout so any arm can be re-derived
from a fresh `adata` copy.

In [1]:
import os
import glob
import warnings
from copy import deepcopy

import anndata as ad
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)

In [2]:
import sys
sys.path.append('../scripts/CapGTA/')
from preprocess_scanpy import gtf_gid_to_symbol

In [3]:
fm._load_fontmanager(try_read_cache=False)
for f in glob.glob(os.path.expanduser('~/.local/share/fonts/texgyreheros*.otf')):
    fm.fontManager.addfont(f)

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['TeX Gyre Heros'],
    'font.size': 12,
    'axes.linewidth': 2.5,
    'xtick.major.width': 2.5,
    'ytick.major.width': 2.5,
    'xtick.minor.width': 2.0,
    'ytick.minor.width': 2.0,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
})

In [4]:
extended_colors = [
    '#2E2A3A',
    '#FF8DAF', '#FF1B5E',
    '#FFEDA0', '#FFDB57',
    '#A4C8FE', '#438CFD',
    '#FFCF80', '#FF8C00',
    '#C4F9AE', '#80F15E',
    '#A0FFFF', '#03FFFF',
    '#A99FD4', '#5941A9',
    '#F4BFFE', '#E980FC',
    '#E8E4F0', '#C4BDD6',
    '#E8C89A', '#C17F3E'
]

drop_donors = ['background', 'worm02', 'worm04', 'worm14', 'worm19']

DNA_OBS_CSV      = '../results/worm6_final/DNA_analysis/obs.csv'
GTF_PATH         = '/shared/biodata/reference/iGenomes/Caenorhabditis_elegans/Ensembl/WBcel235/Annotation/Genes/genes.gtf'
GENE_JUNCTIONS   = '../results/worm6_final/GEX/gene_junctions.csv'
GENE_LENGTHS     = '../results/worm6_final/GEX/gene_lengths.csv'
OUT_DIR          = '../results/worm6_final/GEX/preprocessing_sweep'
os.makedirs(OUT_DIR, exist_ok=True)

## Raw adata inventory

For every (read_source, soupx) pair we hold a raw-count h5ad where
`.X` (and `.layers['counts']` if present) contains integer counts as
emitted by featureCounts (or the SoupX-decontaminated integers from
`reimport_soupx.py`).

The polyat + SoupX chain is submitted as jobs 9600061→9600071 — once
those finish, `adata_gex_polyat_soupx.h5ad` and
`adata_gex_polyat.h5ad` will be present and the polyat arm loaders will
work.

In [5]:
ADATA_PATHS = {
    ('spliced',       False): '../results/worm6_final/GEX/rna_counts/rna_counts.h5ad',
    ('spliced',       True):  '../results/worm6_final/GEX/adata_gex_spliced_soupx.h5ad',
    ('splicedpolyat', False): '../results/worm6_final/GEX/rna_counts_splicedpolyat/rna_counts.h5ad',
    ('splicedpolyat', True):  '../results/worm6_final/GEX/adata_gex_splicedpolyat_soupx.h5ad',
    ('polyat',        False): '../results/worm6_final/GEX/rna_counts_polyat/rna_counts.h5ad',
    ('polyat',        True):  '../results/worm6_final/GEX/adata_gex_polyat_soupx.h5ad',
}

# quick availability check
for k, p in ADATA_PATHS.items():
    print(f'{k[0]:>14s}  soupx={k[1]}  {"OK " if os.path.exists(p) else "-- "} {p}')

       spliced  soupx=False  OK  ../results/worm6_final/GEX/rna_counts/rna_counts.h5ad
       spliced  soupx=True  OK  ../results/worm6_final/GEX/adata_gex_spliced_soupx.h5ad
 splicedpolyat  soupx=False  OK  ../results/worm6_final/GEX/rna_counts_splicedpolyat/rna_counts.h5ad
 splicedpolyat  soupx=True  OK  ../results/worm6_final/GEX/adata_gex_splicedpolyat_soupx.h5ad
        polyat  soupx=False  --  ../results/worm6_final/GEX/rna_counts_polyat/rna_counts.h5ad
        polyat  soupx=True  --  ../results/worm6_final/GEX/adata_gex_polyat_soupx.h5ad


In [6]:
GID2SYM = None  # lazy

def _gid2sym():
    global GID2SYM
    if GID2SYM is None:
        from pathlib import Path
        GID2SYM = gtf_gid_to_symbol(Path(GTF_PATH))
    return GID2SYM


def load_raw(read_source, soupx):
    """Return an AnnData with .X = raw ints, .layers['counts'] = raw ints,
    DNA obs joined, cuticle/failed donors dropped, var_names renamed WBID→symbol,
    n_junctions + exonic_length populated in .var."""
    path = ADATA_PATHS[(read_source, soupx)]
    if not os.path.exists(path):
        raise FileNotFoundError(f'adata missing (pipeline not yet finished?): {path}')
    a = sc.read_h5ad(path)

    # For SoupX'd input, preprocess_scanpy.py wasn't run — so .var_names may still be WBIDs,
    # and .layers['counts'] may already exist. For raw featureCounts input, .X is int, no layer.
    if 'counts' not in a.layers:
        a.layers['counts'] = a.X.copy()

    # WBID → symbol
    gid2sym = _gid2sym()
    if 'wormbase_id' in a.var.columns:
        wbids = a.var['wormbase_id'].astype(str).to_numpy()
    else:
        wbids = a.var_names.to_numpy()
    new_names = np.array([gid2sym.get(g, g) for g in wbids])
    a.var['wormbase_id'] = wbids
    a.var_names = new_names
    a.var_names_make_unique()

    # attach junction / length
    junc = pd.read_csv(GENE_JUNCTIONS).set_index('gene_id')
    lens = pd.read_csv(GENE_LENGTHS).set_index('gene_id')
    a.var['n_junctions']    = junc['n_junctions'].reindex(a.var['wormbase_id']).fillna(0).to_numpy()
    a.var['exonic_length']  = lens['exonic_length'].reindex(a.var['wormbase_id']).fillna(0).to_numpy()
    a.var['exonic_length_kb'] = a.var['exonic_length'] / 1000.0
    a.var['k_JL']           = a.var['n_junctions'] * a.var['exonic_length_kb']

    # attach DNA obs (indexed by barcode)
    dna = pd.read_csv(DNA_OBS_CSV, index_col=0)
    keep = a.obs_names.isin(dna.index)
    a = a[keep].copy()
    a.obs = pd.concat([a.obs, dna.loc[a.obs_names]], axis=1)

    # drop failed donors
    a = a[~a.obs['donor'].isin(drop_donors)].copy()
    return a

## Preprocessing engine

`run_pipeline(adata_raw, config)` — takes a fresh copy, applies the config's
scaling → norm → HVG → regression → PCA/UMAP/leiden, and returns the processed
adata + a metrics dict.

In [7]:
def _apply_scaling(X_sparse, k):
    """Divide columns of a CSR matrix by k (where k>0). Genes with k==0 are left unscaled."""
    if not sp.issparse(X_sparse):
        X_sparse = sp.csr_matrix(X_sparse)
    scale = np.where(k > 0, 1.0 / np.where(k > 0, k, 1.0), 1.0)
    return (X_sparse @ sp.diags(scale)).tocsr()


SCALING_ARMS = {
    'none':      lambda a: a.layers['counts'].astype(np.float32),
    '/J':        lambda a: _apply_scaling(a.layers['counts'].astype(np.float32), a.var['n_junctions'].to_numpy()),
    '/Lkb':      lambda a: _apply_scaling(a.layers['counts'].astype(np.float32), a.var['exonic_length_kb'].to_numpy()),
    '/(J*Lkb)':  lambda a: _apply_scaling(a.layers['counts'].astype(np.float32), a.var['k_JL'].to_numpy()),
}


def _normalize(a, mode):
    if mode == 'standard':
        sc.pp.normalize_total(a, target_sum=1e4)
        sc.pp.log1p(a)
    elif mode == 'dna_size_factor':
        dna = a.obs['unique_aligned_reads'].to_numpy().astype(np.float32)
        sf = dna / np.median(dna)
        X = a.X if sp.issparse(a.X) else sp.csr_matrix(a.X)
        a.X = sp.diags(1.0 / sf) @ X
        sc.pp.log1p(a)
    else:
        raise ValueError(mode)


def _regress(a, mode):
    if mode is None:
        return
    if mode == 'pct_1x':
        sc.pp.regress_out(a, ['pct_1x'], n_jobs=8)
        sc.pp.scale(a, max_value=10)
    elif mode == 'log_unique_aligned_reads':
        a.obs['log_unique_aligned_reads'] = np.log1p(a.obs['unique_aligned_reads'])
        sc.pp.regress_out(a, ['log_unique_aligned_reads'], n_jobs=8)
        sc.pp.scale(a, max_value=10)
    else:
        raise ValueError(mode)


def run_pipeline(adata_raw, config, n_top_genes=2000, resolution=1.0):
    """Return processed adata + metrics dict.

    config keys:
        scaling     : one of SCALING_ARMS
        norm        : 'standard' | 'dna_size_factor'
        regress     : None | 'pct_1x' | 'log_unique_aligned_reads'
    """
    a = adata_raw.copy()
    # --- scaling: replaces .X, leaves .layers['counts'] untouched
    a.X = SCALING_ARMS[config['scaling']](a)
    _normalize(a, config['norm'])
    sc.pp.highly_variable_genes(a, flavor='seurat_v3', n_top_genes=n_top_genes,
                                subset=True, layer='counts')
    _regress(a, config['regress'])

    sc.tl.pca(a, n_comps=50)
    sc.pp.neighbors(a, n_neighbors=15)
    sc.tl.umap(a, min_dist=0.1, spread=5.0)
    sc.tl.leiden(a, resolution=resolution, key_added='leiden_1', flavor='igraph', n_iterations=2)

    # --- DEGs
    sc.tl.rank_genes_groups(a, groupby='leiden_1', method='wilcoxon', use_raw=False)

    # --- PC × DNA-metric correlations
    pcs = a.obsm['X_pca'][:, :30]
    metrics = {}
    for col in ['pct_1x', 'unique_aligned_reads']:
        v = a.obs[col].to_numpy().astype(np.float64)
        r = np.array([np.corrcoef(pcs[:, i], v)[0, 1] for i in range(pcs.shape[1])])
        metrics[col] = r

    return a, metrics


def top_degs(adata_processed, n=100):
    """Return long-form DataFrame of top n DEGs per cluster ranked by log2FC."""
    rg = adata_processed.uns['rank_genes_groups']
    groups = rg['names'].dtype.names
    rows = []
    for g in groups:
        names   = rg['names'][g]
        lfc     = rg['logfoldchanges'][g]
        pvals   = rg['pvals_adj'][g]
        order   = np.argsort(-lfc)[:n]
        for rank, i in enumerate(order):
            rows.append({'cluster': g, 'rank': rank, 'gene': names[i],
                         'log2FC': lfc[i], 'pval_adj': pvals[i]})
    return pd.DataFrame(rows)

## Sweep configs

The full cross-product (3 read-sources × 2 SoupX × 4 scalings × 2 norms × 3 regressions
= 144) is too many to eyeball. Instead we hold most dimensions at a baseline
(`splicedpolyat`, SoupX on, no count scaling, standard norm, no regression) and
vary one dimension at a time.

The dict `SWEEP` is easy to expand — every entry is a full config.

In [13]:
BASELINE = dict(
    read_source='splicedpolyat',
    soupx=True,
    scaling='none',
    norm='standard',
    regress=None,
)


def _cfg(**kw):
    c = dict(BASELINE)
    c.update(kw)
    return c


SWEEP = {
    # baseline
    'baseline_splicedpolyat_soupx':          _cfg(),

    # read-source dimension (holding other arms at baseline)
    'read=spliced_soupx':                    _cfg(read_source='spliced'),
    'read=polyat_soupx':                     _cfg(read_source='polyat'),

    # +/- SoupX (each read source)
    'read=spliced_nosoupx':                  _cfg(read_source='spliced',       soupx=False),
    'read=splicedpolyat_nosoupx':            _cfg(read_source='splicedpolyat', soupx=False),
    'read=polyat_nosoupx':                   _cfg(read_source='polyat',        soupx=False),

    # count-scaling dimension
    'scale=J':                              _cfg(scaling='/J'),
    'scale=Lkb':                            _cfg(scaling='/Lkb'),
    'scale=(J*Lkb)':                        _cfg(scaling='/(J*Lkb)'),

    # normalization dimension
    'norm=dna_size_factor':                  _cfg(norm='dna_size_factor'),

    # regression dimension
    'regress=pct_1x':                        _cfg(regress='pct_1x'),
    'regress=log_uniq_aligned':              _cfg(regress='log_unique_aligned_reads'),

    # a couple of combined "kitchen-sink" runs on each read source
    'polyat_kitchen_sink':                   _cfg(read_source='polyat',
                                                  scaling='/(J*Lkb)', norm='dna_size_factor',
                                                  regress='log_unique_aligned_reads'),
    'splicedpolyat_kitchen_sink':            _cfg(scaling='/(J*Lkb)', norm='dna_size_factor',
                                                  regress='log_unique_aligned_reads'),
}

for name, c in SWEEP.items():
    print(f'{name:40s}  {c}')

baseline_splicedpolyat_soupx              {'read_source': 'splicedpolyat', 'soupx': True, 'scaling': 'none', 'norm': 'standard', 'regress': None}
read=spliced_soupx                        {'read_source': 'spliced', 'soupx': True, 'scaling': 'none', 'norm': 'standard', 'regress': None}
read=polyat_soupx                         {'read_source': 'polyat', 'soupx': True, 'scaling': 'none', 'norm': 'standard', 'regress': None}
read=spliced_nosoupx                      {'read_source': 'spliced', 'soupx': False, 'scaling': 'none', 'norm': 'standard', 'regress': None}
read=splicedpolyat_nosoupx                {'read_source': 'splicedpolyat', 'soupx': False, 'scaling': 'none', 'norm': 'standard', 'regress': None}
read=polyat_nosoupx                       {'read_source': 'polyat', 'soupx': False, 'scaling': 'none', 'norm': 'standard', 'regress': None}
scale=J                                   {'read_source': 'splicedpolyat', 'soupx': True, 'scaling': '/J', 'norm': 'standard', 'regress': None}
sca

In [9]:
# Load each unique (read_source, soupx) pair ONCE and cache.
_RAW_CACHE = {}

def get_raw(read_source, soupx):
    key = (read_source, soupx)
    if key not in _RAW_CACHE:
        print(f'  loading raw adata: {key}')
        _RAW_CACHE[key] = load_raw(read_source, soupx)
    return _RAW_CACHE[key]

## Run the sweep

Each config runs in ~1–3 min (dominated by regress_out when enabled).
Results are stored in `results/` keyed by config name.

In [14]:
results = {}   # name -> {'adata': ..., 'metrics': ..., 'degs': ...}

for name, cfg in SWEEP.items():
    print(f'\n=== {name} ===')
    print(f'  cfg: {cfg}')
    try:
        raw = get_raw(cfg['read_source'], cfg['soupx'])
    except FileNotFoundError as e:
        print(f'  SKIP: {e}')
        continue
    a, metrics = run_pipeline(raw, cfg)
    degs = top_degs(a, n=100)
    degs.to_csv(f'{OUT_DIR}/{name}__top100_degs.csv', index=False)
    results[name] = {'adata': a, 'metrics': metrics, 'degs': degs, 'cfg': cfg}
    print(f'  {a.n_obs} cells x {a.n_vars} genes  |  {a.obs["leiden_1"].nunique()} leiden clusters')
    print(f'  |r|(PC1..30, pct_1x)              max = {np.abs(metrics["pct_1x"]).max():.3f}   '
          f'mean = {np.abs(metrics["pct_1x"]).mean():.3f}')
    print(f'  |r|(PC1..30, unique_aligned_reads) max = {np.abs(metrics["unique_aligned_reads"]).max():.3f}   '
          f'mean = {np.abs(metrics["unique_aligned_reads"]).mean():.3f}')


=== baseline_splicedpolyat_soupx ===
  cfg: {'read_source': 'splicedpolyat', 'soupx': True, 'scaling': 'none', 'norm': 'standard', 'regress': None}


/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/umap/umap_.py:1401: RuntimeWarning: divide by zero encountered in power
  return 1.0 / (1.0 + a * x ** (2 * b))


  1775 cells x 2000 genes  |  9 leiden clusters
  |r|(PC1..30, pct_1x)              max = 0.941   mean = 0.051
  |r|(PC1..30, unique_aligned_reads) max = 0.677   mean = 0.061

=== read=spliced_soupx ===
  cfg: {'read_source': 'spliced', 'soupx': True, 'scaling': 'none', 'norm': 'standard', 'regress': None}


/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/umap/umap_.py:1401: RuntimeWarning: divide by zero encountered in power
  return 1.0 / (1.0 + a * x ** (2 * b))


  1775 cells x 2000 genes  |  11 leiden clusters
  |r|(PC1..30, pct_1x)              max = 0.761   mean = 0.064
  |r|(PC1..30, unique_aligned_reads) max = 0.578   mean = 0.066

=== read=polyat_soupx ===
  cfg: {'read_source': 'polyat', 'soupx': True, 'scaling': 'none', 'norm': 'standard', 'regress': None}
  loading raw adata: ('polyat', True)
  SKIP: adata missing (pipeline not yet finished?): ../results/worm6_final/GEX/adata_gex_polyat_soupx.h5ad

=== read=spliced_nosoupx ===
  cfg: {'read_source': 'spliced', 'soupx': False, 'scaling': 'none', 'norm': 'standard', 'regress': None}


...

/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/umap/umap_.py:1401: RuntimeWarning: divide by zero encountered in power
  return 1.0 / (1.0 + a * x ** (2 * b))
/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:484: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:484: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:484: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:484: RuntimeWarning: invalid value encoun

  1775 cells x 2000 genes  |  10 leiden clusters
  |r|(PC1..30, pct_1x)              max = 0.446   mean = 0.039
  |r|(PC1..30, unique_aligned_reads) max = 0.041   mean = 0.015

=== polyat_kitchen_sink ===
  cfg: {'read_source': 'polyat', 'soupx': True, 'scaling': '/(J*Lkb)', 'norm': 'dna_size_factor', 'regress': 'log_unique_aligned_reads'}
  loading raw adata: ('polyat', True)
  SKIP: adata missing (pipeline not yet finished?): ../results/worm6_final/GEX/adata_gex_polyat_soupx.h5ad

=== splicedpolyat_kitchen_sink ===
  cfg: {'read_source': 'splicedpolyat', 'soupx': True, 'scaling': '/(J*Lkb)', 'norm': 'dna_size_factor', 'regress': 'log_unique_aligned_reads'}


/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/umap/umap_.py:1401: RuntimeWarning: divide by zero encountered in power
  return 1.0 / (1.0 + a * x ** (2 * b))


  1775 cells x 2000 genes  |  11 leiden clusters
  |r|(PC1..30, pct_1x)              max = 0.163   mean = 0.038
  |r|(PC1..30, unique_aligned_reads) max = 0.034   mean = 0.012


/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:484: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:484: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:484: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:484: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/dmullane/micromamba/envs/default_jupyter/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:484: Runt

## UMAP grid — leiden clusters per config

Each panel: UMAP with points colored by leiden cluster (extended palette,
matching the existing worm6 GEX notebook).

In [ ]:
def plot_umap_grid(results, cols=3, s=25, savepath=None):
    names = list(results.keys())
    rows = int(np.ceil(len(names) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.5 * rows))
    axes = np.atleast_2d(axes).ravel()
    for ax, name in zip(axes, names):
        a = results[name]['adata']
        sc.pl.umap(a, color='leiden_1', ax=ax, show=False, frameon=False,
                   palette=extended_colors, s=s, legend_loc=None,
                   title=name, colorbar_loc=None)
    for ax in axes[len(names):]:
        ax.axis('off')
    plt.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=200, bbox_inches='tight')
    return fig

plot_umap_grid(results, cols=3, savepath=f'{OUT_DIR}/sweep_umap_grid.png');

### Same grid, colored by `pct_1x` — visual read on how much DNA coverage drives structure

In [ ]:
def plot_umap_grid_metric(results, metric, cmap='viridis', cols=3, s=25, savepath=None):
    names = list(results.keys())
    rows = int(np.ceil(len(names) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.5 * rows))
    axes = np.atleast_2d(axes).ravel()
    for ax, name in zip(axes, names):
        a = results[name]['adata']
        sc.pl.umap(a, color=metric, ax=ax, show=False, frameon=False,
                   cmap=cmap, s=s, title=f'{name}\n{metric}', colorbar_loc=None)
    for ax in axes[len(names):]:
        ax.axis('off')
    plt.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=200, bbox_inches='tight')
    return fig

plot_umap_grid_metric(results, 'pct_1x', savepath=f'{OUT_DIR}/sweep_umap_grid_pct1x.png');

In [ ]:
plot_umap_grid_metric(results, 'unique_aligned_reads',
                      savepath=f'{OUT_DIR}/sweep_umap_grid_uniq_aligned.png');

## PC × DNA library-size correlation

For each config we computed Pearson r between PC 1..30 and each of `pct_1x`
and `unique_aligned_reads`. Below: heatmap of |r| across configs (rows) and
PCs (cols). Bright rows = a lot of the PC variance is DNA-driven; dark rows
= DNA structure has been washed out (either because the arm removed it, or
because the arm removed all signal).

In [ ]:
def build_correlation_frame(results, which='pct_1x'):
    rows = []
    for name, r in results.items():
        rows.append(pd.Series(np.abs(r['metrics'][which]), name=name))
    df = pd.concat(rows, axis=1).T
    df.columns = [f'PC{i+1}' for i in range(df.shape[1])]
    return df

df_pct1x = build_correlation_frame(results, 'pct_1x')
df_uniq  = build_correlation_frame(results, 'unique_aligned_reads')

fig, axes = plt.subplots(2, 1, figsize=(12, 0.4 * len(results) + 2))
for ax, (df, title) in zip(axes, [(df_pct1x, '|r|(PC, pct_1x)'),
                                   (df_uniq,  '|r|(PC, unique_aligned_reads)')]):
    sns.heatmap(df, ax=ax, cmap='magma', vmin=0, vmax=0.6, cbar_kws={'label': '|r|'})
    ax.set_title(title)
    ax.set_xlabel('')
plt.tight_layout()
fig.savefig(f'{OUT_DIR}/pc_dna_correlation_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

# summary table: max |r| and sum(|r|) across first 30 PCs
summary = pd.DataFrame({
    'max|r|_pct1x':   df_pct1x.max(axis=1),
    'sum|r|_pct1x':   df_pct1x.sum(axis=1),
    'max|r|_uniq':    df_uniq.max(axis=1),
    'sum|r|_uniq':    df_uniq.sum(axis=1),
})
summary.to_csv(f'{OUT_DIR}/pc_dna_correlation_summary.csv')
summary.sort_values('max|r|_pct1x')

## DEG comparison

For each config, a wide DataFrame of the top-100 DEGs per cluster is saved
to `{OUT_DIR}/{name}__top100_degs.csv`. Quick summary below: number of
clusters and the median (across clusters) of the top-1 DEG's log2FC — a
rough proxy for how strongly clusters are separated by their marker genes.

In [ ]:
summary_degs = []
for name, r in results.items():
    d = r['degs']
    top1 = d[d['rank'] == 0]
    summary_degs.append({
        'config':                 name,
        'n_clusters':             d['cluster'].nunique(),
        'median_top1_log2FC':     top1['log2FC'].median(),
        'p95_top1_log2FC':        top1['log2FC'].quantile(0.95),
    })
summary_degs = pd.DataFrame(summary_degs).set_index('config')
summary_degs.to_csv(f'{OUT_DIR}/deg_summary.csv')
summary_degs

## Notes / follow-ups

- `polyat_*` configs will `SKIP` in the run cell until the SLURM pipeline
  chain (jobs 9600061 → 9600071) finishes writing
  `rna_counts_polyat/rna_counts.h5ad` and `adata_gex_polyat_soupx.h5ad`.
  Just re-run the sweep cell once they land.
- All raw counts are preserved in `.layers['counts']` throughout, so if you
  want to add another arm (e.g. scVI, PearsonResiduals), you can grab
  `results[name]['adata']` and go from there.
- DNA obs columns available for regression include: `pct_1x`, `pct_5x`,
  `pct_10x`, `mean_coverage`, `median_coverage`, `unique_aligned_reads`,
  `total_reads`, `aligned_reads`, `pct_duplicates`, `gini` — swap any of
  these into the regress arm to test more hypotheses about which DNA
  variable is doing the damage.